In [51]:
#Step 1
#Install mlxtend
#!pip install mlxtend

#Load libraries
import pandas as pd
import seaborn as sns
from mlxtend.frequent_patterns import apriori, association_rules



In [52]:
# STEP 2: Load and prepare Titanic data
df = pd.read_excel('/content/sample_data/AssociationRule.xlsx'
, sheet_name='Sheet1',usecols=['TOTAL_FINANCE_RATE_ratio', 'DISCOUNT_ratio','QUOTEDRV_ratio','COST_MAINT_PER_MONTH_ratio','PREMIUM_ratio','NEW_STATUS'] )
# Now you can work with the DataFrame 'df'
df.head()

,NEW_STATUS,TOTAL_FINANCE_RATE_ratio,DISCOUNT_ratio,QUOTEDRV_ratio,COST_MAINT_PER_MONTH_ratio,PREMIUM_ratio
0,Not Converted,High,Low,Medium-High,Medium-High,Medium-High
1,Not Converted,High,Low,Low,High,Medium-High
2,Not Converted,High,Low,Medium-High,Medium-High,High
3,Not Converted,High,Low,Medium-High,Medium-Low,Low
4,Not Converted,High,Low,Medium-Low,High,High


Note: parch = Parent/Children abroad,  sibsp = Siblings/Spouse Aboard, deck = cabin deck letter, alone = traveling alone

In [53]:
#STEP 3: One-hot encode for Apriori
quote_ohe = pd.get_dummies(df[['TOTAL_FINANCE_RATE_ratio', 'DISCOUNT_ratio','QUOTEDRV_ratio','COST_MAINT_PER_MONTH_ratio','PREMIUM_ratio','NEW_STATUS']])
quote_ohe.head()

,TOTAL_FINANCE_RATE_ratio_High,TOTAL_FINANCE_RATE_ratio_Low,TOTAL_FINANCE_RATE_ratio_Medium-High,TOTAL_FINANCE_RATE_ratio_Medium-Low,DISCOUNT_ratio_High,DISCOUNT_ratio_Low,DISCOUNT_ratio_Medium,QUOTEDRV_ratio_High,QUOTEDRV_ratio_Low,QUOTEDRV_ratio_Medium-High,...,COST_MAINT_PER_MONTH_ratio_High,COST_MAINT_PER_MONTH_ratio_Low,COST_MAINT_PER_MONTH_ratio_Medium-High,COST_MAINT_PER_MONTH_ratio_Medium-Low,PREMIUM_ratio_High,PREMIUM_ratio_Low,PREMIUM_ratio_Medium-High,PREMIUM_ratio_Medium-Low,NEW_STATUS_Converted,NEW_STATUS_Not Converted
0,True,False,False,False,False,True,False,False,False,True,...,False,False,True,False,False,False,True,False,False,True
1,True,False,False,False,False,True,False,False,True,False,...,True,False,False,False,False,False,True,False,False,True
2,True,False,False,False,False,True,False,False,False,True,...,False,False,True,False,True,False,False,False,False,True
3,True,False,False,False,False,True,False,False,False,True,...,False,False,False,True,False,True,False,False,False,True
4,True,False,False,False,False,True,False,False,False,False,...,True,False,False,False,True,False,False,False,False,True


In [93]:
#Step 4
# Run Apriori
frequent_itemsets = apriori(quote_ohe, min_support=0.05, use_colnames=True)
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.6)

# Sort and show top rules
top_rules = rules.sort_values(by='lift', ascending=False).head(10)
top_rules[['antecedents', 'consequents', 'support', 'confidence', 'lift', 'conviction']]

,antecedents,consequents,support,confidence,lift,conviction
579,"(PREMIUM_ratio_Medium-Low, TOTAL_FINANCE_RATE_...","(COST_MAINT_PER_MONTH_ratio_Low, DISCOUNT_rati...",0.065359,0.975610,13.569845,38.052288
590,"(COST_MAINT_PER_MONTH_ratio_Low, DISCOUNT_rati...","(PREMIUM_ratio_Medium-Low, TOTAL_FINANCE_RATE_...",0.065359,0.909091,13.569845,10.263072
589,"(QUOTEDRV_ratio_Medium-Low, TOTAL_FINANCE_RATE...","(COST_MAINT_PER_MONTH_ratio_Low, PREMIUM_ratio...",0.065359,0.869565,13.304348,7.165577
580,"(COST_MAINT_PER_MONTH_ratio_Low, PREMIUM_ratio...","(QUOTEDRV_ratio_Medium-Low, TOTAL_FINANCE_RATE...",0.065359,1.000000,13.304348,inf
407,"(TOTAL_FINANCE_RATE_ratio_Low, QUOTEDRV_ratio_...","(COST_MAINT_PER_MONTH_ratio_Low, PREMIUM_ratio...",0.066993,0.891304,12.685541,8.553595
406,"(COST_MAINT_PER_MONTH_ratio_Low, PREMIUM_ratio...","(TOTAL_FINANCE_RATE_ratio_Low, QUOTEDRV_ratio_...",0.066993,0.953488,12.685541,19.883987
584,"(QUOTEDRV_ratio_Medium-Low, TOTAL_FINANCE_RATE...","(COST_MAINT_PER_MONTH_ratio_Low, PREMIUM_ratio...",0.065359,0.888889,12.651163,8.367647
587,"(COST_MAINT_PER_MONTH_ratio_Low, PREMIUM_ratio...","(QUOTEDRV_ratio_Medium-Low, TOTAL_FINANCE_RATE...",0.065359,0.930233,12.651163,13.279412
456,"(COST_MAINT_PER_MONTH_ratio_Low, DISCOUNT_rati...","(PREMIUM_ratio_Medium-Low, QUOTEDRV_ratio_Medi...",0.065359,0.909091,12.363636,10.191176
457,"(PREMIUM_ratio_Medium-Low, QUOTEDRV_ratio_Medi...","(COST_MAINT_PER_MONTH_ratio_Low, DISCOUNT_rati...",0.065359,0.888889,12.363636,8.352941


In [98]:
#STEP 5: Strong rule mining
frequent_itemsets = apriori(quote_ohe, min_support=0.0001, use_colnames=True) #0.1
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.8)

#Filter interesting rules
strong_rules = rules[rules['lift'] > 2.0].sort_values(by=['support','lift'], ascending=[False,False])

#Display top rules
strong_rules[['antecedents', 'consequents', 'support', 'confidence', 'lift','conviction']].head(10)


,antecedents,consequents,support,confidence,lift,conviction
10,(NEW_STATUS_Converted),(COST_MAINT_PER_MONTH_ratio_Low),0.253268,0.880682,2.763986,5.710551
184,"(DISCOUNT_ratio_Medium, COST_MAINT_PER_MONTH_r...",(PREMIUM_ratio_Low),0.241830,0.986667,3.946667,56.250000
185,(PREMIUM_ratio_Low),"(DISCOUNT_ratio_Medium, COST_MAINT_PER_MONTH_r...",0.241830,0.967320,3.946667,23.100000
183,"(PREMIUM_ratio_Low, DISCOUNT_ratio_Medium)",(COST_MAINT_PER_MONTH_ratio_Low),0.241830,1.000000,3.138462,inf
9,(PREMIUM_ratio_Low),(COST_MAINT_PER_MONTH_ratio_Low),0.241830,0.967320,3.035897,20.850000
1237,"(PREMIUM_ratio_Low, NEW_STATUS_Converted)","(DISCOUNT_ratio_Medium, COST_MAINT_PER_MONTH_r...",0.225490,1.000000,4.080000,inf
1240,"(DISCOUNT_ratio_Medium, COST_MAINT_PER_MONTH_r...","(PREMIUM_ratio_Low, NEW_STATUS_Converted)",0.225490,0.920000,4.080000,9.681373
1235,"(DISCOUNT_ratio_Medium, COST_MAINT_PER_MONTH_r...",(PREMIUM_ratio_Low),0.225490,1.000000,4.000000,inf
1242,(PREMIUM_ratio_Low),"(DISCOUNT_ratio_Medium, COST_MAINT_PER_MONTH_r...",0.225490,0.901961,4.000000,7.900000
1236,"(PREMIUM_ratio_Low, COST_MAINT_PER_MONTH_ratio...","(DISCOUNT_ratio_Medium, NEW_STATUS_Converted)",0.225490,0.932432,3.705511,11.075817


In [106]:
#STEP 6 Mining rules of interest Converted

#frequent_itemsets = apriori(titanic_ohe, min_support=0.01, use_colnames=True)

#Force RHS to be of "convert"
convert_rules = rules[
   (rules['consequents'].apply(lambda x: 'NEW_STATUS_Converted' in x)) &
    (rules['consequents'].apply(lambda x: len(x) == 1)) # ระบุจำนวน consequents ที่ต้องการ
    &(rules['antecedents'].apply(lambda x: len(x) == 3)) # ระบุจำนวน antecedents ที่ต้องการ
]
convert_rules = convert_rules.sort_values(by=['support'],ascending=[False])
convert_rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(10)


,antecedents,consequents,support,confidence,lift
1233,"(PREMIUM_ratio_Low, COST_MAINT_PER_MONTH_ratio...",(NEW_STATUS_Converted),0.225490,0.932432,3.242322
1290,"(PREMIUM_ratio_Low, QUOTEDRV_ratio_Medium-High...",(NEW_STATUS_Converted),0.148693,0.947917,3.296165
1192,"(DISCOUNT_ratio_Medium, QUOTEDRV_ratio_Medium-...",(NEW_STATUS_Converted),0.148693,0.938144,3.262184
1201,"(PREMIUM_ratio_Low, QUOTEDRV_ratio_Medium-High...",(NEW_STATUS_Converted),0.148693,0.947917,3.296165
643,"(PREMIUM_ratio_Low, QUOTEDRV_ratio_Medium-High...",(NEW_STATUS_Converted),0.147059,0.937500,3.259943
567,"(DISCOUNT_ratio_Medium, COST_MAINT_PER_MONTH_r...",(NEW_STATUS_Converted),0.147059,0.927835,3.226336
577,"(PREMIUM_ratio_Low, TOTAL_FINANCE_RATE_ratio_L...",(NEW_STATUS_Converted),0.147059,0.947368,3.294258
629,"(COST_MAINT_PER_MONTH_ratio_Low, TOTAL_FINANCE...",(NEW_STATUS_Converted),0.147059,0.937500,3.259943
549,"(DISCOUNT_ratio_Medium, QUOTEDRV_ratio_Medium-...",(NEW_STATUS_Converted),0.147059,0.882353,3.068182
676,"(PREMIUM_ratio_Low, COST_MAINT_PER_MONTH_ratio...",(NEW_STATUS_Converted),0.147059,0.947368,3.294258


In [107]:
#STEP 7 Mining rules of interest Not Converted

#frequent_itemsets = apriori(titanic_ohe, min_support=0.01, use_colnames=True)

#Force RHS to be of "convert"
not_convert_rules = rules[
   (rules['consequents'].apply(lambda x: 'NEW_STATUS_Not Converted' in x)) &
    (rules['consequents'].apply(lambda x: len(x) == 1)) # ระบุจำนวน consequents ที่ต้องการ
    &(rules['antecedents'].apply(lambda x: len(x) == 3)) # ระบุจำนวน antecedents ที่ต้องการ
]
not_convert_rules = not_convert_rules.sort_values(by='support', ascending=False)
not_convert_rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(10)

,antecedents,consequents,support,confidence,lift
307,"(DISCOUNT_ratio_Low, TOTAL_FINANCE_RATE_ratio_...",(NEW_STATUS_Not Converted),0.101307,1.000000,1.403670
320,"(DISCOUNT_ratio_Low, TOTAL_FINANCE_RATE_ratio_...",(NEW_STATUS_Not Converted),0.099673,1.000000,1.403670
1142,"(DISCOUNT_ratio_Low, COST_MAINT_PER_MONTH_rati...",(NEW_STATUS_Not Converted),0.093137,1.000000,1.403670
1103,"(DISCOUNT_ratio_Low, COST_MAINT_PER_MONTH_rati...",(NEW_STATUS_Not Converted),0.088235,1.000000,1.403670
408,"(TOTAL_FINANCE_RATE_ratio_High, COST_MAINT_PER...",(NEW_STATUS_Not Converted),0.086601,1.000000,1.403670
287,"(DISCOUNT_ratio_Low, TOTAL_FINANCE_RATE_ratio_...",(NEW_STATUS_Not Converted),0.066993,1.000000,1.403670
731,"(DISCOUNT_ratio_Low, TOTAL_FINANCE_RATE_ratio_...",(NEW_STATUS_Not Converted),0.063725,1.000000,1.403670
1115,"(DISCOUNT_ratio_Low, PREMIUM_ratio_Medium-High...",(NEW_STATUS_Not Converted),0.062092,1.000000,1.403670
1149,"(DISCOUNT_ratio_Low, PREMIUM_ratio_Medium-High...",(NEW_STATUS_Not Converted),0.060458,1.000000,1.403670
1120,"(DISCOUNT_ratio_Low, QUOTEDRV_ratio_Medium-Hig...",(NEW_STATUS_Not Converted),0.058824,0.947368,1.329792


In [108]:
#STEP 8 Mining rules of interest Converted all antecedents

#frequent_itemsets = apriori(quote_ohe, min_support=0.01, use_colnames=True)

#Force RHS to be of "convert"
convert_rules = rules[(rules['antecedents'].apply(lambda x:
                                                  any('TOTAL_FINANCE_RATE_ratio' in str(i) for i in x) and
                                                  any('DISCOUNT_ratio' in str(i) for i in x) and
                                                  any('QUOTEDRV_ratio' in str(i) for i in x) and
                                                  any('COST_MAINT_PER_MONTH_ratio' in str(i) for i in x) and
                                                  any('PREMIUM_ratio' in str(i) for i in x)))
   & (rules['consequents'].apply(lambda x: any('NEW_STATUS' in str(i) for i in x)))
    & (rules['consequents'].apply(lambda x: len(x) == 1))
]
convert_rules = convert_rules.sort_values(['consequents','support'],ascending=[False, False])
convert_rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(10)

,antecedents,consequents,support,confidence,lift
3685,"(DISCOUNT_ratio_Medium, QUOTEDRV_ratio_Medium-...",(NEW_STATUS_Converted),0.147059,0.947368,3.294258
4250,"(DISCOUNT_ratio_Medium, PREMIUM_ratio_Low, TOT...",(NEW_STATUS_Converted),0.044118,0.870968,3.028592
3989,"(DISCOUNT_ratio_Medium, PREMIUM_ratio_Low, TOT...",(NEW_STATUS_Converted),0.032680,0.952381,3.311688
3842,"(PREMIUM_ratio_High, COST_MAINT_PER_MONTH_rati...",(NEW_STATUS_Converted),0.001634,1.000000,3.477273
4192,"(DISCOUNT_ratio_Medium, QUOTEDRV_ratio_Medium-...",(NEW_STATUS_Converted),0.001634,1.000000,3.477273
4221,"(DISCOUNT_ratio_Medium, QUOTEDRV_ratio_Medium-...",(NEW_STATUS_Converted),0.001634,1.000000,3.477273
3255,"(COST_MAINT_PER_MONTH_ratio_High, PREMIUM_rati...",(NEW_STATUS_Not Converted),0.031046,1.000000,1.403670
3323,"(COST_MAINT_PER_MONTH_ratio_High, PREMIUM_rati...",(NEW_STATUS_Not Converted),0.022876,1.000000,1.403670
3482,"(QUOTEDRV_ratio_High, COST_MAINT_PER_MONTH_rat...",(NEW_STATUS_Not Converted),0.019608,1.000000,1.403670
3827,"(PREMIUM_ratio_Medium-High, COST_MAINT_PER_MON...",(NEW_STATUS_Not Converted),0.019608,1.000000,1.403670
